# Vet Clinics in Berlin – OSMNX Data Extraction

This notebook fetches veterinary clinics in Berlin directly from OpenStreetMap
via OSMNX. It replaces the previous manual OSM export and provides a repeatable,
up-to-date snapshot of vet clinics (`amenity = veterinary`) within Berlin.

The output of this notebook is used as the OSM input for:

- `01_vet_clinics_osm_lor_join.ipynb` (spatial join with LOR / districts),
- `02_vet_clinics_cleaning_and_normalization.ipynb` (cleaned v1 dataset).

In [1]:
import osmnx as ox
import geopandas as gpd
from pathlib import Path

# OSMNX settings: enable local cache and log to console
ox.settings.log_console = True
ox.settings.use_cache = True

ox.__version__

'2.0.7'

## 1. Fetch vet clinics from OSM using OSMNX

We use `osmnx.features.features_from_place` to download all OSM features
tagged as `amenity = veterinary` within the administrative boundary of
"Berlin, Germany".

This returns a GeoDataFrame with OSM tags (name, address, contact info,
opening hours, etc.) and geometries (points, lines or polygons).

In [2]:
import osmnx as ox

place_name = "Berlin, Germany"

# OSM filter: vet clinics
tags = {"amenity": "veterinary"}

gdf_osm = ox.features.features_from_place(place_name, tags=tags)

print(f"Number of OSM features fetched: {len(gdf_osm)}")
gdf_osm.head()

Number of OSM features fetched: 175


geometry addr:city addr:housenumber  \
element id                                                                
node    268917040  POINT (13.40523 52.49568)    Berlin               69   
        299795048  POINT (13.47955 52.60629)    Berlin               67   
        347294456  POINT (13.32013 52.42972)    Berlin               36   
        394867279   POINT (13.27057 52.5352)       NaN              NaN   
        411550894  POINT (13.58964 52.50951)    Berlin               19   

                  addr:postcode          addr:street     amenity  \
element id                                                         
node    268917040         10961       Baerwaldstraße  veterinary   
        299795048         13125            Straße 48  veterinary   
        347294456         12207  Königsberger Straße  veterinary   
        394867279           NaN                  NaN  veterinary   
        411550894         12621        Planitzstraße  veterinary   

                                                   name  \
element id                                                
node    268917040               Tierarztpraxis am Urban   
        299795048            Dr. med. vet. Elke Hartwig   
        347294456  Tierarztpraxis Dr. Bernhard Sörensen   
        394867279       Tierarztpraxis Jeanette Koepsel   
        411550894  Kleintierarztpraxis Berlin Kaulsdorf   

                                          note  \
element id                                       
node    268917040  Städt. Kita Baerwaldstr. 69   
        299795048                          NaN   
        347294456                          NaN   
        394867279                          NaN   
        411550894                          NaN   

                                                       opening_hours  \
element id                                                             
node    268917040  Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...   
        299795048         Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00   
        347294456              Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00   
        394867279                                                NaN   
        411550894  Mo-Fr 09:00-19:00 open "tel. Terminvereinbarun...   

                  wheelchair  ... heritage heritage:operator lda:criteria  \
element id                    ...                                           
node    268917040         no  ...      NaN               NaN          NaN   
        299795048    limited  ...      NaN               NaN          NaN   
        347294456        yes  ...      NaN               NaN          NaN   
        394867279        NaN  ...      NaN               NaN          NaN   
        411550894        NaN  ...      NaN               NaN          NaN   

                  old_name ref:lda wikidata wikimedia_commons roof:levels  \
element id                                                                  
node    268917040      NaN     NaN      NaN               NaN         NaN   
        299795048      NaN     NaN      NaN               NaN         NaN   
        347294456      NaN     NaN      NaN               NaN         NaN   
        394867279      NaN     NaN      NaN               NaN         NaN   
        411550894      NaN     NaN      NaN               NaN         NaN   

                  historic source:shape  
element id                               
node    268917040      NaN          NaN  
        299795048      NaN          NaN  
        347294456      NaN          NaN  
        394867279      NaN          NaN  
        411550894      NaN          NaN  

[5 rows x 77 columns]

## 2. Normalize geometry and compute `lat` / `lon`

To work consistently with point locations, we:

1. Ensure the GeoDataFrame has a CRS (WGS84 / EPSG:4326).
2. Project to a metric CRS (Web Mercator, EPSG:3857) for accurate centroids.
3. Compute centroids in the projected CRS.
4. Re-project centroids back to WGS84 (EPSG:4326).
5. Store these centroids as the main geometry and expose `lat` and `lon`
   columns for downstream processing.

In [3]:
# 1) Ensure the CRS is set (OSM data is typically in EPSG:4326)
if gdf_osm.crs is None:
    gdf_osm = gdf_osm.set_crs(epsg=4326)

# 2) Project to a metric CRS for centroid calculation
gdf_proj = gdf_osm.to_crs(epsg=3857)

# 3) Compute centroids in the projected CRS
centroids_proj = gdf_proj.geometry.centroid

# 4) Re-project centroids back to WGS84
centroids_wgs84 = centroids_proj.to_crs(epsg=4326)

# 5) Overwrite geometry with WGS84 centroids and extract lat/lon
gdf_osm["geometry"] = centroids_wgs84
gdf_osm["lat"] = gdf_osm.geometry.y
gdf_osm["lon"] = gdf_osm.geometry.x

gdf_osm[["name", "addr:street", "addr:housenumber", "lat", "lon"]].head()

name          addr:street  \
element id                                                                     
node    268917040               Tierarztpraxis am Urban       Baerwaldstraße   
        299795048            Dr. med. vet. Elke Hartwig            Straße 48   
        347294456  Tierarztpraxis Dr. Bernhard Sörensen  Königsberger Straße   
        394867279       Tierarztpraxis Jeanette Koepsel                  NaN   
        411550894  Kleintierarztpraxis Berlin Kaulsdorf        Planitzstraße   

                  addr:housenumber        lat        lon  
element id                                                
node    268917040               69  52.495684  13.405233  
        299795048               67  52.606286  13.479555  
        347294456               36  52.429722  13.320133  
        394867279              NaN  52.535199  13.270573  
        411550894               19  52.509511  13.589635

## 3. Select relevant attributes and define `source_osm_id`

We keep a focused subset of OSM tags that are relevant for the vet clinics
layer and the downstream cleaning / modelling notebooks.

In addition, we create a `source_osm_id` field for traceability back to
the original OSM element. Depending on the OSMNX version, this may come
from the `osmid` column or from the (element, id) index.

In [4]:
import geopandas as gpd

# Relevant attributes we want to keep for the vet clinics pipeline
cols_keep = [
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "phone",
    "contact:phone",
    "email",
    "contact:email",
    "website",
    "contact:website",
    "opening_hours",
    "operator",
    "wheelchair",
    "wheelchair:description",
    "emergency",
    "lat",
    "lon",
]

# Source OSM id for traceability
if "osmid" in gdf_osm.columns:
    gdf_osm["source_osm_id"] = gdf_osm["osmid"].astype(str)
else:
    # `features_from_place` often returns a MultiIndex (element, id)
    # Flatten it into a string identifier
    gdf_osm["source_osm_id"] = gdf_osm.index.to_flat_index().astype(str)

# Final column order: id + geometry + selected attributes (only those that exist)
cols_export = ["source_osm_id", "geometry"] + [
    c for c in cols_keep if c in gdf_osm.columns
]

# Build a proper GeoDataFrame subset with geometry and CRS preserved
gdf_osm_subset = gpd.GeoDataFrame(
    gdf_osm[cols_export].copy(),
    geometry="geometry",
    crs=gdf_osm.crs,
)

gdf_osm_subset.head()

source_osm_id                   geometry  \
element id                                                          
node    268917040  ('node', 268917040)  POINT (13.40523 52.49568)   
        299795048  ('node', 299795048)  POINT (13.47955 52.60629)   
        347294456  ('node', 347294456)  POINT (13.32013 52.42972)   
        394867279  ('node', 394867279)   POINT (13.27057 52.5352)   
        411550894  ('node', 411550894)  POINT (13.58964 52.50951)   

                                                   name          addr:street  \
element id                                                                     
node    268917040               Tierarztpraxis am Urban       Baerwaldstraße   
        299795048            Dr. med. vet. Elke Hartwig            Straße 48   
        347294456  Tierarztpraxis Dr. Bernhard Sörensen  Königsberger Straße   
        394867279       Tierarztpraxis Jeanette Koepsel                  NaN   
        411550894  Kleintierarztpraxis Berlin Kaulsdorf        Planitzstraße   

                  addr:housenumber addr:postcode addr:city            phone  \
element id                                                                    
node    268917040               69         10961    Berlin              NaN   
        299795048               67         13125    Berlin   +49 30 9437820   
        347294456               36         12207    Berlin   +49 30 7738321   
        394867279              NaN           NaN       NaN              NaN   
        411550894               19         12621    Berlin  +49 30 53018585   

                  contact:phone                       email contact:email  \
element id                                                                  
node    268917040           NaN                         NaN           NaN   
        299795048           NaN                         NaN           NaN   
        347294456           NaN                         NaN           NaN   
        394867279           NaN                         NaN           NaN   
        411550894           NaN  info@tierarzt-kaulsdorf.de           NaN   

                                                    website contact:website  \
element id                                                                    
node    268917040                                       NaN             NaN   
        299795048     http://www.tierarztpraxis-hartwig.de/             NaN   
        347294456  https://www.tierarztpraxis-soerensen.de/             NaN   
        394867279                                       NaN             NaN   
        411550894        https://www.tierarzt-kaulsdorf.de/             NaN   

                                                       opening_hours  \
element id                                                             
node    268917040  Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...   
        299795048         Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00   
        347294456              Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00   
        394867279                                                NaN   
        411550894  Mo-Fr 09:00-19:00 open "tel. Terminvereinbarun...   

                                              operator wheelchair  \
element id                                                          
node    268917040                                  NaN         no   
        299795048                                  NaN    limited   
        347294456                                  NaN        yes   
        394867279                                  NaN        NaN   
        411550894  Dr. Berit Miels;Dr. Mathias Kochert        NaN   

                  wheelchair:description emergency        lat        lon  
element id                                                                
node    268917040                    NaN       NaN  52.495684  13.405233  
        299795048                    NaN       NaN  52.606286  13.479555  
        347294456                    NaN   

## 4. Export OSM vet clinics snapshot (GeoJSON + CSV)

We export the resulting dataset in two formats:

- **GeoJSON** under `veterinary_clinics/sources/`:
  used as the main spatial input for the join with LOR (notebook 01).

- **CSV** under `veterinary_clinics/cache/`:
  used for quick inspection or non-spatial processing.

The notebook is located inside the `veterinary_clinics/` directory, so
we use relative paths `sources/...` and `cache/...`.

In [5]:
from pathlib import Path

# NOTE: current working directory is expected to be
# .../layered-populate-data-pool-da/veterinary_clinics

output_geojson = Path("sources/osmnx_berlin_vet_clinics_latest.geojson")
output_csv = Path("cache/osmnx_berlin_vet_clinics_latest.csv")

# Ensure the target directories exist
output_geojson.parent.mkdir(parents=True, exist_ok=True)
output_csv.parent.mkdir(parents=True, exist_ok=True)

# 1) GeoJSON keeps geometry
gdf_osm_subset.to_file(output_geojson, driver="GeoJSON")

# 2) CSV export without geometry (attributes + lat/lon)
gdf_osm_subset.drop(columns="geometry").to_csv(output_csv, index=False)

print("Exported:")
print(" -", output_geojson)
print(" -", output_csv)

Exported:
 - sources/osmnx_berlin_vet_clinics_latest.geojson
 - cache/osmnx_berlin_vet_clinics_latest.csv


## 5. Quick sanity check of exported files (optional)

We verify that the exported GeoJSON and CSV files exist on disk. This
step is optional but helpful to confirm that the snapshot is available
for `01_vet_clinics_osm_lor_join.ipynb`.

In [6]:
from pathlib import Path

print("GeoJSON exists? ", Path("sources/osmnx_berlin_vet_clinics_latest.geojson").exists())
print("CSV exists?     ", Path("cache/osmnx_berlin_vet_clinics_latest.csv").exists())

GeoJSON exists?  True
CSV exists?      True
